# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Técnicas de aprendizaje profundo · Laboratorio ilustrado UTP
 **Cristian Alvarado · Guía de práctica para el diplomado**

 Un recorrido desde los datos hasta redes neuronales, visión, texto y generación. Cada bloque tiene problema, código, gráfico y una modificación para explorar.

 **Cómo usarlo en Colab:** Archivo → Guardar una copia en Drive; ejecuta primero las dos celdas de preparación. Después usa el índice lateral para ir al caso de la guía. Los bloques comparten las funciones de preparación, pero crean sus propios datos: no necesitan CSV externos, claves de API ni GPU.

 **Datos:** todos los casos de café, fincas y mensajes son simulados para aprender; no son modelos agronómicos validados. Para visión utilizamos los dígitos de scikit-learn, incluidos en la biblioteca. Los laboratorios de atención y difusión son modelos pequeños, no implementaciones de GPT o generadores fotográficos.

 **Ruta sugerida:** 01–03 fundamentos; 04–08 aprendizaje automático; 09–13 aprendizaje profundo; 14 proyecto. La guía es un menú de talleres: no es necesario ejecutarlos todos en una sesión.

 **Regla del taller:** predice qué ocurrirá, ejecuta, cambia una variable y explica lo que observas. No vale responder «lo dijo Python».


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn', 'xgboost':'xgboost-cpu',
            'lightgbm':'lightgbm', 'shap':'shap'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


## 01. Matemáticas: del dato al cálculo
 **El problema:** Una neurona recibe uniformidad del color 0,6 y proporción de manchas 0,2. ¿Cómo obtiene una puntuación?

 **Vamos a hacer:** Calcular producto punto, media y desviación estándar; recorrer una curva de error con descenso del gradiente.

 **Antes de ejecutar:** ¿Qué cambia si el peso de las manchas pasa de −1 a −2?

 **Cómo comprobarlo:** La puntuación inicial es 0,5; al cambiar ese peso pasa a 0,3. Los pesos no son medidas de similitud.


In [ ]:
entradas = np.array([.6,.2]); pesos = np.array([2.,-1.]); sesgo = -.5
contribuciones = entradas * pesos
z = entradas @ pesos + sesgo
print('Contribuciones:',contribuciones,'· sesgo:',sesgo,'· puntuación:',z)
rendimientos = np.array([28,30,25,33,29])
print('Media:',rendimientos.mean(),'· varianza:',rendimientos.var(),
      '· desviación estándar:',rendimientos.std())
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].bar(['Color × peso','Manchas × peso','Sesgo'],[*contribuciones,sesgo],color=[AZUL,VERDE,GRIS]);ax[0].axhline(0,color='black',lw=.7);ax[0].set(title=f'La suma es z = {z:.1f}',ylabel='Aporte a la puntuación')
ax[1].bar(range(1,6),rendimientos,color=VERDE);ax[1].axhline(rendimientos.mean(),color=AZUL,ls='--',label='Media');ax[1].set(title='Promedio y variación',xlabel='Lote',ylabel='Rendimiento ilustrativo');ax[1].legend()
mostrar('01-matematicas')


In [ ]:
# Pérdida L(w) = (w−3)^2. Su derivada es 2(w−3).
TASA = .15 # prueba .02 o 1.1 y compara, no siempre converge
w = -2.; camino = [w]
for paso in range(12):
    w = w - TASA*2*(w-3)
    camino.append(w)
curva=np.linspace(-3,5,120)
plt.plot(curva,(curva-3)**2,color=AZUL);plt.plot(camino,(np.array(camino)-3)**2,'o-',color=VERDE)
plt.title('Descenso del gradiente: pasos hacia menor error');plt.xlabel('Peso w');plt.ylabel('Pérdida (w−3)²');mostrar('01-gradiente')
print('Peso final:',round(w,3),'· error final:',round((w-3)**2,4))


## 02. Python: una herramienta por tarea
 **El problema:** Tenemos rendimientos de cuatro lotes y queremos resumirlos sin calcular a mano.

 **Vamos a hacer:** Usar lista, función, NumPy y una tabla de Pandas.

 **Antes de ejecutar:** Añade un quinto lote de rendimiento 50. ¿Qué ocurrirá con la media?

 **Cómo comprobarlo:** La media original es 2,5; al agregar 50 aumenta mucho. Un valor extremo puede distorsionar el resumen.


In [ ]:
rendimientos = [1,2,3,4]
def promedio(valores):
    return sum(valores)/len(valores)
tabla = pd.DataFrame({'Lote':['A','B','C','D'],'Rendimiento':rendimientos})
display(tabla)
print('Con Python:',promedio(rendimientos),'· con NumPy:',np.mean(rendimientos))
tabla.plot.bar(x='Lote',y='Rendimiento',color=AZUL,legend=False)
plt.ylabel('Rendimiento ilustrativo');plt.title('Cada fila es un lote; cada columna, una variable');mostrar('02-python')


## 03. Datos y EDA: mirar antes de modelar
 **El problema:** Faltan lecturas de humedad y las variables tienen unidades muy distintas.

 **Vamos a hacer:** Crear un CSV, visualizar, separar entrenamiento/prueba e imputar y escalar dentro de un Pipeline.

 **Antes de ejecutar:** ¿Qué significa humedad = 0,8 después de escalar? ¿Es 80 % de humedad física?

 **Cómo comprobarlo:** No: es una posición dentro del rango aprendido en entrenamiento. Los parámetros de limpieza se aprenden sin mirar la prueba.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
X,y=datos_fincas()
X.loc[X.index[::13],'Humedad_pct']=np.nan
archivo=CARPETA/'fincas_simuladas.csv'
X.assign(Rendimiento_t_ha=y).to_csv(archivo,index=False)
leido=pd.read_csv(archivo);display(leido.head())
print('Faltantes por columna:\n',X.isna().sum())
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEMILLA)
modelo=make_pipeline(SimpleImputer(strategy='median'),MinMaxScaler(),LinearRegression())
modelo.fit(Xtr,ytr);pred=modelo.predict(Xte)
print('MAE en prueba:',round(mean_absolute_error(yte,pred),3),'t/ha')
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].hist(X.Humedad_pct.dropna(),bins=12,color=AZUL);ax[0].set(title='¿Cómo se distribuye la humedad?',xlabel='Humedad (%)',ylabel='Número de fincas')
ax[1].scatter(X.Temperatura_C,y,color=VERDE,alpha=.6);ax[1].set(title='Una relación que conviene explorar',xlabel='Temperatura (°C)',ylabel='Rendimiento simulado (t/ha)');mostrar('03-eda')
print('NDVI: índice de vegetación, no diagnóstico automático de salud.')
ndvi=np.array([.65,.72,.58,.75,.60,.81,.70]);print('Desviación estándar NDVI:',round(ndvi.std(),3))


## 04. Regresión: predecir una cantidad
 **El problema:** ¿Cómo se relacionan el área de una vivienda y su precio? ¿Qué pasa cuando la relación es curva?

 **Vamos a hacer:** Comparar línea, polinomio, Ridge y Lasso en datos nuevos; mostrar MAE y R².

 **Antes de ejecutar:** Cambia GRADO de 2 a 12. ¿La curva más flexible mejora siempre fuera del entrenamiento?

 **Cómo comprobarlo:** No. Más flexibilidad puede ajustar el ruido. Compara errores de entrenamiento y prueba, sin elegir el modelo mirando repetidamente la prueba.


In [ ]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.preprocessing import PolynomialFeatures
r=np.random.default_rng(SEMILLA)
area=r.uniform(40,180,100);precio=40+2*area+.008*(area-90)**2+r.normal(0,18,100)
X=area[:,None];Xtr,Xte,ytr,yte=train_test_split(X,precio,test_size=.25,random_state=SEMILLA)
GRADO=2
modelos={'Lineal':LinearRegression(),
         'Polinomio':make_pipeline(PolynomialFeatures(GRADO,include_bias=False),StandardScaler(),LinearRegression()),
         'Ridge':make_pipeline(PolynomialFeatures(GRADO,include_bias=False),StandardScaler(),Ridge(alpha=1)),
         'Lasso':make_pipeline(PolynomialFeatures(GRADO,include_bias=False),StandardScaler(),Lasso(alpha=.5,max_iter=5000))}
resultados=[];malla=np.linspace(40,180,150)[:,None]
plt.scatter(Xtr[:,0],ytr,color=GRIS,label='Entrenamiento',alpha=.6)
plt.scatter(Xte[:,0],yte,color='black',marker='x',label='Prueba')
for nombre,m in modelos.items():
    m.fit(Xtr,ytr);p=m.predict(Xte)
    resultados.append([nombre,mean_absolute_error(ytr,m.predict(Xtr)),mean_absolute_error(yte,p),r2_score(yte,p)])
    plt.plot(malla[:,0],m.predict(malla),label=nombre)
display(pd.DataFrame(resultados,columns=['Modelo','MAE entrenamiento','MAE prueba','R² prueba']).round(3))
plt.xlabel('Área (m²)');plt.ylabel('Precio ilustrativo (millones COP)');plt.legend();plt.title('Predecir un número, no una categoría');mostrar('04-regresion')


## 05. Agrupación: encontrar grupos sin etiquetas
 **El problema:** Queremos explorar perfiles de parcelas y concentraciones de puntos de observación.

 **Vamos a hacer:** Comparar K-Means, DBSCAN y OPTICS con gráficos de colores.

 **Antes de ejecutar:** Cambia K de 3 a 4 y EPS de 8 a 3. ¿Los colores son diagnósticos agronómicos?

 **Cómo comprobarlo:** No: son grupos matemáticos. DBSCAN marca puntos no agrupados como −1; no demuestra que sean errores o plagas. OPTICS sí devuelve labels_.


In [ ]:
from sklearn.cluster import KMeans,DBSCAN,OPTICS
from sklearn.datasets import make_blobs
K=3;EPS=8
puntos,_=make_blobs(n_samples=[35,35,35],centers=[(15,20),(65,70),(80,20)],cluster_std=[2,5,3],random_state=SEMILLA)
puntos=np.vstack([puntos,[[2,90],[95,90],[45,45]]]) # coordenadas en metros
modelos={'K-Means':KMeans(n_clusters=K,random_state=SEMILLA,n_init=10),
         'DBSCAN':DBSCAN(eps=EPS,min_samples=4),
         'OPTICS':OPTICS(min_samples=5,xi=.05,min_cluster_size=.1)}
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,(nombre,m) in zip(axs,modelos.items()):
    etiquetas=m.fit_predict(puntos)
    ax.scatter(puntos[:,0],puntos[:,1],c=etiquetas,cmap='tab10',s=30)
    ax.set(title=nombre,xlabel='X (m)',ylabel='Y (m)');ax.set_aspect('equal')
    print(nombre,'· grupos:',len(set(etiquetas)-{-1}),'· no agrupados:',sum(etiquetas==-1))
mostrar('05-clustering')
# Para mezclar rendimiento y humedad, estandariza las variables antes de K-Means.
# Para coordenadas métricas en el mismo sistema, conservamos metros y EPS también en metros.


## 06. Clasificación: asignar una etiqueta
 **El problema:** A partir de color y manchas simulados, queremos clasificar un grano como sano o defectuoso.

 **Vamos a hacer:** Entrenar KNN, SVM, árbol, Naive Bayes y regresión logística con la misma separación y dibujar sus fronteras.

 **Antes de ejecutar:** Cambia VECINOS de 3 a 15. ¿Se vuelve más suave la frontera de KNN?

 **Cómo comprobarlo:** Las fronteras muestran reglas aprendidas. El resultado en prueba estima desempeño en estos datos simulados; no valida inspección de café real.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
X,y=datos_cafe();Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
VECINOS=3
modelos={'KNN':make_pipeline(StandardScaler(),KNeighborsClassifier(VECINOS)),
         'SVM':make_pipeline(StandardScaler(),SVC(kernel='rbf',C=1)),
         'Árbol':DecisionTreeClassifier(max_depth=3,random_state=SEMILLA),
         'Naive Bayes':GaussianNB(),
         'Logística':make_pipeline(StandardScaler(),LogisticRegression(random_state=SEMILLA))}
xx,yy=np.meshgrid(np.linspace(0,1,90),np.linspace(0,1,90));malla=pd.DataFrame(np.c_[xx.ravel(),yy.ravel()],columns=X.columns)
fig,axs=plt.subplots(2,3,figsize=(12,7));filas=[]
for ax,(nombre,m) in zip(axs.ravel(),modelos.items()):
    m.fit(Xtr,ytr);p=m.predict(Xte);filas.append([nombre,accuracy_score(yte,p)])
    ax.contourf(xx,yy,m.predict(malla).reshape(xx.shape),levels=[-.5,.5,1.5],colors=['#f8e8dc','#e1f1e9'],alpha=.8)
    ax.scatter(Xte.iloc[:,0],Xte.iloc[:,1],c=yte,cmap='RdYlGn',edgecolor='black',s=25)
    ax.set(title=nombre,xlabel='Uniformidad del color',ylabel='Proporción de manchas')
axs.ravel()[-1].axis('off');mostrar('06-clasificacion')
display(pd.DataFrame(filas,columns=['Modelo','Accuracy prueba']).round(3))
print('Ejemplo nuevo: sano=1, defectuoso=0')
nuevo=pd.DataFrame([[.6,.2]],columns=X.columns)
print('Probabilidad estimada con logística:',modelos['Logística'].predict_proba(nuevo))


## 07. Ensamblados: varios modelos, una predicción
 **El problema:** Un árbol puede cambiar mucho si cambian los datos. ¿Ayuda combinar varios?

 **Vamos a hacer:** Comparar árbol, bagging, Random Forest, AdaBoost, XGBoost y LightGBM usando las mismas fincas y la misma métrica.

 **Antes de ejecutar:** Reduce n_estimators a 5. ¿Todos empeoran necesariamente?

 **Cómo comprobarlo:** No existe un ganador universal. Bagging combina modelos independientes; boosting agrega modelos secuenciales. Esta comparación utiliza regresión, no etiquetas.


In [ ]:
from sklearn.ensemble import BaggingRegressor,RandomForestRegressor,AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
X,y=datos_fincas();Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEMILLA)
ARBOLES=40
modelos={'Promedio (baseline)':DummyRegressor(),
         'Un árbol':DecisionTreeRegressor(max_depth=4,random_state=SEMILLA),
         'Bagging':BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=4),n_estimators=ARBOLES,random_state=SEMILLA,n_jobs=1),
         'Random Forest':RandomForestRegressor(n_estimators=ARBOLES,max_depth=6,random_state=SEMILLA,n_jobs=1),
         'AdaBoost':AdaBoostRegressor(n_estimators=ARBOLES,random_state=SEMILLA),
         'XGBoost':XGBRegressor(n_estimators=ARBOLES,max_depth=3,learning_rate=.1,random_state=SEMILLA,n_jobs=1),
         'LightGBM':LGBMRegressor(n_estimators=ARBOLES,num_leaves=7,min_child_samples=10,verbosity=-1,random_state=SEMILLA,n_jobs=1)}
filas=[]
for nombre,m in modelos.items():
    m.fit(Xtr,ytr);p=m.predict(Xte);filas.append([nombre,mean_absolute_error(yte,p)])
resultados=pd.DataFrame(filas,columns=['Modelo','MAE prueba']).sort_values('MAE prueba');display(resultados.round(3))
plt.barh(resultados.Modelo,resultados['MAE prueba'],color=AZUL);plt.xlabel('Error absoluto medio (t/ha) · menor es mejor');plt.title('Comparación en el mismo conjunto de prueba');mostrar('07-ensambles')


## 08. Evaluar y explicar: cuánto falla y dónde
 **El problema:** Un detector omite una plaga y también produce una falsa alarma. ¿Cómo contamos esos errores?

 **Vamos a hacer:** Construir matriz de confusión; calcular métricas; validar con grupos y tiempo; descomponer una predicción con SHAP.

 **Antes de ejecutar:** Sube el umbral de .5 a .8 en el ejemplo de probabilidades. ¿Qué pasa con las detecciones?

 **Cómo comprobarlo:** Recall pregunta cuántos positivos reales detectamos; precision pregunta cuántas alertas eran correctas. SHAP explica el modelo, no demuestra causas.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay,precision_score,recall_score,f1_score,roc_auc_score
verdad=np.array([1,0,1,1,0,0]);prob=np.array([.9,.1,.3,.8,.2,.7]);UMBRAL=.5
pred=(prob>=UMBRAL).astype(int)
ConfusionMatrixDisplay.from_predictions(verdad,pred,display_labels=['Sin plaga','Con plaga'],cmap='Blues',colorbar=False)
plt.title('¿Qué error produce cada casilla?');mostrar('08-metricas')
print('Accuracy:',accuracy_score(verdad,pred))
print('Precision:',precision_score(verdad,pred,zero_division=0))
print('Recall:',recall_score(verdad,pred,zero_division=0))
print('F1:',f1_score(verdad,pred,zero_division=0),'· ROC-AUC:',roc_auc_score(verdad,prob))
print('Con umbral 0,5: TP=2, FN=1, FP=1, TN=2; recall=2/3, precision=2/3.')


In [ ]:
from sklearn.model_selection import KFold,GroupKFold,TimeSeriesSplit,cross_val_score
from sklearn.ensemble import RandomForestRegressor
X,y=datos_fincas(n=120);grupos=np.repeat(np.arange(12),10)
reg=RandomForestRegressor(n_estimators=30,max_depth=5,random_state=SEMILLA,n_jobs=1)
cv=KFold(5,shuffle=True,random_state=SEMILLA)
mae=-cross_val_score(reg,X,y,cv=cv,scoring='neg_mean_absolute_error')
print('MAE por partición aleatoria:',np.round(mae,3),'· media:',round(mae.mean(),3))
# Demostración de separación: si hay varias filas por finca, mantenlas juntas.
train_g,test_g=next(GroupKFold(3).split(X,y,groups=grupos))
print('Fincas compartidas entre train y test:',set(grupos[train_g])&set(grupos[test_g]))
train_t,test_t=next(TimeSeriesSplit(3).split(np.arange(40)))
print('Ejemplo temporal: entrenar hasta',train_t[-1],'y evaluar desde',test_t[0])
# SHAP para una predicción de regresión; aquí sí ejecutamos el explicador.
import shap
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEMILLA)
reg.fit(Xtr,ytr);explicador=shap.TreeExplainer(reg);explicacion=explicador(Xte.iloc[:1])
base=float(np.asarray(explicacion.base_values).ravel()[0]);aportes=explicacion.values[0];pred=float(reg.predict(Xte.iloc[:1])[0])
print('Referencia:',round(base,3),'· aportes:',np.round(aportes,3),'· predicción:',round(pred,3))
assert np.isclose(base+aportes.sum(),pred,atol=1e-5)
plt.barh(X.columns,aportes,color=[VERDE if v>0 else AZUL for v in aportes]);plt.axvline(0,color='black',lw=.7);plt.xlabel('Contribución a ESTA predicción (t/ha)');plt.title('SHAP: referencia + contribuciones = predicción');mostrar('08-shap')


## 09. Redes neuronales: de una neurona al aprendizaje
 **El problema:** Primero seguimos la neurona del café; después entrenamos una pequeña red con ejemplos etiquetados.

 **Vamos a hacer:** Mostrar pesos, sesgo y activaciones; entrenar con mini-lotes; ver pérdida y frontera.

 **Antes de ejecutar:** Cambia TASA a .001 o TAMANO_LOTE a 16. ¿Cómo cambia la curva de aprendizaje?

 **Cómo comprobarlo:** Backpropagation calcula gradientes; el optimizador actualiza parámetros. Una época recorre el entrenamiento, un lote solo una parte.


In [ ]:
# Una neurona con parámetros explícitos. No está entrenada: reproduce la guía.
capa=nn.Linear(2,1)
with torch.no_grad():
    capa.weight.copy_(torch.tensor([[2.,-1.]]));capa.bias.copy_(torch.tensor([-.5]))
x=torch.tensor([[.6,.2]])
z=capa(x);p=torch.sigmoid(z)
print('Entradas:',x.tolist(),'· pesos:',capa.weight.detach().tolist(),'· sesgo:',capa.bias.item())
print('Puntuación:',z.item(),'· P(sano):',p.item())
valores=torch.linspace(-4,4,150)
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,(nombre,f) in zip(axs,[('Identidad',lambda x:x),('Sigmoide',torch.sigmoid),('tanh',torch.tanh),('ReLU',torch.relu)]):
    ax.plot(valores,f(valores),color=AZUL);ax.axhline(0,color=GRIS,lw=.7);ax.set(title=nombre,xlabel='z',ylabel='f(z)')
mostrar('09-activaciones')
print('Softmax [2,1,0]:',torch.softmax(torch.tensor([2.,1.,0.]),dim=0).tolist())


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
X,y=datos_cafe(n=320);Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
escalador=StandardScaler().fit(Xtr)
xt=torch.tensor(escalador.transform(Xtr),dtype=torch.float32);yt=torch.tensor(ytr[:,None],dtype=torch.float32)
xe=torch.tensor(escalador.transform(Xte),dtype=torch.float32)
TASA=.02;TAMANO_LOTE=32;EPOCAS=40
torch.manual_seed(SEMILLA)
red=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,8),nn.ReLU(),nn.Linear(8,1))
perdida=nn.BCEWithLogitsLoss() # recibe puntuaciones; no añadir sigmoide antes
opt=torch.optim.Adam(red.parameters(),lr=TASA)
lotes=DataLoader(TensorDataset(xt,yt),batch_size=TAMANO_LOTE,shuffle=True)
historial=[]
for epoca in range(EPOCAS):
    red.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(red(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    historial.append(suma/len(xt))
red.eval()
with torch.no_grad():probs=torch.sigmoid(red(xe)).numpy().ravel()
print('Accuracy en prueba:',accuracy_score(yte,probs>=.5))
fig,axs=plt.subplots(1,2,figsize=(10,4));axs[0].plot(historial,color=AZUL);axs[0].set(title='Aprender: reducir el error',xlabel='Época',ylabel='Pérdida de entrenamiento')
xx,yy=np.meshgrid(np.linspace(0,1,80),np.linspace(0,1,80));malla=pd.DataFrame(np.c_[xx.ravel(),yy.ravel()],columns=X.columns)
with torch.no_grad():mapa=torch.sigmoid(red(torch.tensor(escalador.transform(malla),dtype=torch.float32))).numpy().reshape(xx.shape)
axs[1].contourf(xx,yy,mapa,levels=np.linspace(0,1,11),cmap='RdYlGn');axs[1].scatter(Xte.iloc[:,0],Xte.iloc[:,1],c=yte,cmap='RdYlGn',edgecolor='black',s=20);axs[1].set(title='Predecir: usar los pesos aprendidos',xlabel='Uniformidad del color',ylabel='Proporción de manchas');mostrar('09-red-entrenada')


## 10. Visión y CNN: aprender de píxeles
 **El problema:** Queremos reconocer dígitos escritos a mano y distinguir clasificación, detección y segmentación.

 **Vamos a hacer:** Ver un filtro de bordes y entrenar una CNN pequeña sobre dígitos 8×8.

 **Antes de ejecutar:** ¿Qué pasa si EPOCAS baja de 12 a 1? ¿Reconocer un dígito permite ubicar varios objetos?

 **Cómo comprobarlo:** Son tareas distintas. Entrenamos clasificación; las cajas y máscaras de la segunda visualización solo ilustran otras salidas, no un detector entrenado.


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
D=load_digits();X=D.images.astype('float32')/16;y=D.target
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
xt=torch.tensor(Xtr[:,None]);yt=torch.tensor(ytr,dtype=torch.long);xe=torch.tensor(Xte[:,None])
torch.manual_seed(SEMILLA)
cnn=nn.Sequential(nn.Conv2d(1,8,kernel_size=3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),nn.Linear(8*4*4,10))
opt=torch.optim.Adam(cnn.parameters(),lr=.01);perdida=nn.CrossEntropyLoss();lotes=DataLoader(TensorDataset(xt,yt),batch_size=64,shuffle=True)
EPOCAS=12;hist=[]
for epoca in range(EPOCAS):
    cnn.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(cnn(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(xt))
cnn.eval()
with torch.no_grad():pred=cnn(xe).argmax(1).numpy()
print('CNN · accuracy en prueba:',round(accuracy_score(yte,pred),3))
fig,axs=plt.subplots(2,6,figsize=(11,4))
for i,ax in enumerate(axs.ravel()):
    ax.imshow(Xte[i],cmap='gray');ax.set_title(f'Real {yte[i]} → pred. {pred[i]}',fontsize=9);ax.axis('off')
mostrar('10-cnn')
# Convolución con filtro fijo para visualizar la operación (no es un filtro aprendido).
filtro=torch.tensor([[[[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]]])
with torch.no_grad():bordes=torch.nn.functional.conv2d(xe[:1],filtro,padding=1).numpy()[0,0]
fig,axs=plt.subplots(1,3,figsize=(10,3));axs[0].imshow(Xte[0],cmap='gray');axs[0].set_title('Entrada: 8×8 píxeles');axs[1].imshow(bordes,cmap='coolwarm');axs[1].set_title('Filtro: cambios horizontales');axs[2].plot(hist,color=AZUL);axs[2].set(title='Error de la CNN',xlabel='Época',ylabel='Pérdida');mostrar('10-convolucion')


In [ ]:
# Ilustración exacta de tres TIPOS de salida, usando objetos sintéticos.
from matplotlib.patches import Rectangle
h,w=60,90;fila,col=np.mgrid[:h,:w]
mascara=((fila-30)**2+(col-25)**2<13**2)|((fila-28)**2+(col-65)**2<10**2)
imagen=np.ones((h,w,3))*.93;imagen[mascara]=[.2,.65,.4]
fig,axs=plt.subplots(1,3,figsize=(11,3))
for ax in axs:ax.axis('off')
axs[0].imshow(imagen);axs[0].set_title('Clasificación: «hay objetos»')
axs[1].imshow(imagen);axs[1].add_patch(Rectangle((12,17),26,26,fill=False,ec='red',lw=2));axs[1].add_patch(Rectangle((55,18),20,20,fill=False,ec='red',lw=2));axs[1].set_title('Detección: ¿dónde están?')
axs[2].imshow(mascara,cmap='Greens');axs[2].set_title('Segmentación: ¿qué píxeles?');mostrar('10-tareas-vision')
print('Estas cajas y máscaras se dibujaron a partir de la geometría: no ejecutan YOLO ni U-Net.')


## 11. Texto y secuencias: contar palabras no basta
 **El problema:** Debemos organizar solicitudes escritas; además, una secuencia de sensor cambia con el tiempo.

 **Vamos a hacer:** Comparar Bag of Words y TF-IDF; clasificar solicitudes; ver una representación reducida y entrenar una LSTM en una serie simulada.

 **Antes de ejecutar:** Compara «hay plaga» y «no hay plaga». ¿Qué información pierde un conteo de palabras?

 **Cómo comprobarlo:** Los conteos ignoran el orden; TF-IDF pondera frecuencia, no entiende por sí solo la negación. La LSTM recibe una ventana ordenada y se evalúa en el futuro, sin barajar el tiempo.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
textos=['factura pago pendiente','pago factura del proveedor','factura recibida para pago','solicito pago de factura',
        'revisar contrato firmado','firma del contrato pendiente','contrato requiere revisión','solicito revisar contrato',
        'pedido entrega retrasada','entrega del pedido pendiente','pedido llegó incompleto','solicito rastrear pedido']
etiquetas=['factura']*4+['contrato']*4+['pedido']*4
v=CountVectorizer();conteos=v.fit_transform(['hay plaga','no hay plaga','plaga hay'])
display(pd.DataFrame(conteos.toarray(),columns=v.get_feature_names_out(),index=['hay plaga','no hay plaga','plaga hay']))
Xtr,Xte,ytr,yte=train_test_split(textos,etiquetas,test_size=.25,stratify=etiquetas,random_state=SEMILLA)
clasificador=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=1000))
clasificador.fit(Xtr,ytr);print('Accuracy en solo 3 mensajes de prueba:',accuracy_score(yte,clasificador.predict(Xte)))
print('Nuevo mensaje:',clasificador.predict(['por favor enviar la factura para pago'])[0])
tfidf=TfidfVectorizer().fit_transform(textos);coords=TruncatedSVD(2,random_state=SEMILLA).fit_transform(tfidf)
for cat,c in zip(['factura','contrato','pedido'],[AZUL,VERDE,GRIS]):
    idx=np.array(etiquetas)==cat;plt.scatter(coords[idx,0],coords[idx,1],label=cat,s=55)
plt.legend();plt.xlabel('Componente 1');plt.ylabel('Componente 2');plt.title('Representación reducida de textos (TF-IDF + SVD, no Word2Vec)');mostrar('11-textos')


In [ ]:
# Una LSTM real para una secuencia pequeña y artificial.
r=np.random.default_rng(SEMILLA);tiempo=np.arange(240);serie=np.sin(tiempo*.12)+.12*r.normal(size=len(tiempo))
VENTANA=12;CORTE=180
entradas=np.array([serie[i-VENTANA:i] for i in range(VENTANA,len(serie))],dtype='float32')
objetivos=serie[VENTANA:].astype('float32');instante=tiempo[VENTANA:]
train=instante<CORTE;test=~train
xt=torch.tensor(entradas[train,:,None]);yt=torch.tensor(objetivos[train,None]);xe=torch.tensor(entradas[test,:,None])
class PredictorLSTM(nn.Module):
    def __init__(self):
        super().__init__();self.memoria=nn.LSTM(1,12,batch_first=True);self.salida=nn.Linear(12,1)
    def forward(self,x):
        secuencia,_=self.memoria(x);return self.salida(secuencia[:,-1])
torch.manual_seed(SEMILLA);lstm=PredictorLSTM();opt=torch.optim.Adam(lstm.parameters(),lr=.02)
for epoca in range(90):
    opt.zero_grad();error=nn.functional.mse_loss(lstm(xt),yt);error.backward();opt.step()
lstm.eval()
with torch.no_grad():p=lstm(xe).numpy().ravel()
baseline=entradas[test,-1]
print('MAE LSTM:',round(mean_absolute_error(objetivos[test],p),3),'· MAE último valor:',round(mean_absolute_error(objetivos[test],baseline),3))
plt.plot(instante[test],objetivos[test],label='Real simulado',color=GRIS);plt.plot(instante[test],p,label='LSTM',color=AZUL);plt.plot(instante[test],baseline,label='Último valor',color=VERDE,alpha=.6)
plt.legend();plt.xlabel('Tiempo');plt.ylabel('Lectura normalizada');plt.title('Pronóstico de un paso usando la ventana disponible');mostrar('11-lstm')
print('Se evalúa un paso hacia adelante; no es un pronóstico recursivo de toda la serie.')


## 12. Transformers: atención y orden
 **El problema:** Una palabra necesita contexto; una secuencia no significa lo mismo cuando cambiamos su orden.

 **Vamos a hacer:** Calcular atención con números visibles y entrenar un Transformer pequeño que compare el primero y último elemento.

 **Antes de ejecutar:** Invierte una secuencia [1,2,3,4]. La clase debe cambiar. ¿Cambiaría un simple conteo de sus números?

 **Cómo comprobarlo:** No cambiaría el conteo. La atención mezcla información y la posición permite distinguir el orden. El modelo del taller no es BERT ni GPT.


In [ ]:
# Atención explicada con tres fichas y vectores ilustrativos, no semántica aprendida.
fichas=['café','camión','averiado']
Q=np.array([[1.,0.],[0.,1.],[0.,1.]])
K=Q.copy();V=np.array([[1.,0.],[0.,1.],[0.,2.]])
puntuaciones=Q@K.T/np.sqrt(2)
exp=np.exp(puntuaciones-puntuaciones.max(axis=1,keepdims=True));atencion=exp/exp.sum(axis=1,keepdims=True)
resultado=atencion@V
print('Cada fila suma:',atencion.sum(axis=1),'· salida mezclada:\n',np.round(resultado,3))
plt.imshow(atencion,cmap='Blues',vmin=0,vmax=1);plt.xticks(range(3),fichas);plt.yticks(range(3),fichas)
for i in range(3):
    for j in range(3):plt.text(j,i,f'{atencion[i,j]:.2f}',ha='center',va='center')
plt.xlabel('¿De quién toma información?');plt.ylabel('¿Quién consulta?');plt.title('Atención: puntuaciones → softmax → mezcla');mostrar('12-atencion')


In [ ]:
# Transformer real, pequeño, entrenado para una tarea de orden.
r=np.random.default_rng(SEMILLA);seq=r.integers(1,8,size=(700,4));seq=seq[seq[:,0]!=seq[:,-1]]
y=(seq[:,0]>seq[:,-1]).astype('int64')
Xtr,Xte,ytr,yte=train_test_split(seq,y,test_size=.25,stratify=y,random_state=SEMILLA)
class OrdenTransformer(nn.Module):
    def __init__(self):
        super().__init__();self.emb=nn.Embedding(8,16);self.pos=nn.Parameter(torch.randn(1,4,16)*.1)
        capa=nn.TransformerEncoderLayer(d_model=16,nhead=2,dim_feedforward=32,dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(capa,num_layers=1,enable_nested_tensor=False);self.salida=nn.Linear(4*16,2)
    def forward(self,x):
        h=self.encoder(self.emb(x)+self.pos);return self.salida(h.flatten(1))
torch.manual_seed(SEMILLA);transformer=OrdenTransformer();opt=torch.optim.Adam(transformer.parameters(),lr=.005)
loader=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=64,shuffle=True)
hist=[]
for epoca in range(45):
    transformer.train();suma=0
    for bx,by in loader:
        opt.zero_grad();error=nn.functional.cross_entropy(transformer(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(Xtr))
transformer.eval()
with torch.no_grad():
    pred=transformer(torch.tensor(Xte)).argmax(1).numpy()
    ejemplos=torch.tensor([[1,2,3,4],[4,3,2,1]]);prob=torch.softmax(transformer(ejemplos),dim=1).numpy()
print('Accuracy en prueba:',round(accuracy_score(yte,pred),3));print('P(primero > último), originales e invertidos:',np.round(prob[:,1],3))
plt.plot(hist,color=AZUL);plt.xlabel('Época');plt.ylabel('Pérdida');plt.title('Aprender un patrón donde el orden importa');mostrar('12-transformer')


## 13. Visión avanzada: reconstruir y generar
 **El problema:** Una imagen está dañada por ruido. ¿Podemos aprender a reconstruirla? ¿Qué significa aprender a quitar ruido para generar?

 **Vamos a hacer:** Entrenar un autoencoder sobre dígitos y un modelo de difusión pequeño sobre puntos 2D.

 **Antes de ejecutar:** Aumenta RUIDO de .25 a .5. ¿La reconstrucción recupera todos los detalles?

 **Cómo comprobarlo:** No: puede suavizar o inventar detalles. Un autoencoder reconstruye; la difusión entrenada genera muestras. Los puntos 2D no son imágenes fotográficas.


In [ ]:
# Autoencoder de eliminación de ruido: aprende entrada dañada → objetivo limpio.
D=load_digits();X=D.data.astype('float32')/16
Xtr,Xte=train_test_split(X,test_size=.25,random_state=SEMILLA)
xt=torch.tensor(Xtr);xe=torch.tensor(Xte)
torch.manual_seed(SEMILLA);ae=nn.Sequential(nn.Linear(64,24),nn.ReLU(),nn.Linear(24,16),nn.ReLU(),nn.Linear(16,24),nn.ReLU(),nn.Linear(24,64),nn.Sigmoid())
opt=torch.optim.Adam(ae.parameters(),lr=.01);RUIDO=.35
loader=DataLoader(TensorDataset(xt),batch_size=64,shuffle=True)
for epoca in range(100):
    for (limpio,) in loader:
        dañado=torch.clamp(limpio+RUIDO*torch.randn_like(limpio),0,1)
        opt.zero_grad();error=nn.functional.mse_loss(ae(dañado),limpio);error.backward();opt.step()
ae.eval();torch.manual_seed(77);ruidoso=torch.clamp(xe[:6]+RUIDO*torch.randn_like(xe[:6]),0,1)
with torch.no_grad():recon=ae(ruidoso)
fig,axs=plt.subplots(3,6,figsize=(10,5))
for fila,(datos,titulo) in enumerate([(xe[:6],'Original'),(ruidoso,'Con ruido'),(recon,'Reconstrucción')]):
    for i,ax in enumerate(axs[fila]):ax.imshow(datos[i].numpy().reshape(8,8),cmap='gray',vmin=0,vmax=1);ax.axis('off');ax.set_title(titulo if i==0 else '',fontsize=10)
mostrar('13-autoencoder')
print('MSE ruidoso:',nn.functional.mse_loss(ruidoso,xe[:6]).item(),'· MSE reconstruido:',nn.functional.mse_loss(recon,xe[:6]).item())
print('No es SAM, DINOv2 ni un Masked Autoencoder de Transformers; es un autoencoder pequeño entrenado aquí.')


In [ ]:
# Difusión DDPM en 2D: añadir ruido y aprender a predecirlo.
# Dos grupos artificiales. No requiere descargar un modelo grande.
torch.manual_seed(SEMILLA);n=600
centros=torch.tensor([[-1.5,0.],[1.5,0.]])
limpios=centros[torch.arange(n)%2]+.3*torch.randn(n,2)
PASOS=50;beta=torch.linspace(.0005,.08,PASOS);alpha=1-beta;acumulado=torch.cumprod(alpha,0)
class QuitaRuido(nn.Module):
    def __init__(self):
        super().__init__();self.red=nn.Sequential(nn.Linear(3,48),nn.SiLU(),nn.Linear(48,48),nn.SiLU(),nn.Linear(48,2))
    def forward(self,x,t):return self.red(torch.cat([x,t[:,None].float()/(PASOS-1)],dim=1))
modelo_ruido=QuitaRuido();opt=torch.optim.Adam(modelo_ruido.parameters(),lr=.002)
ITERACIONES=900
for i in range(ITERACIONES):
    x0=limpios[torch.randint(0,n,(128,))];t=torch.randint(0,PASOS,(128,));ruido=torch.randn_like(x0)
    abar=acumulado[t,None];xt=abar.sqrt()*x0+(1-abar).sqrt()*ruido
    opt.zero_grad();error=nn.functional.mse_loss(modelo_ruido(xt,t),ruido);error.backward();opt.step()
modelo_ruido.eval();x=torch.randn(600,2)
with torch.no_grad():
    for paso in reversed(range(PASOS)):
        t=torch.full((len(x),),paso,dtype=torch.long);eps=modelo_ruido(x,t)
        media=(x-beta[paso]/torch.sqrt(1-acumulado[paso])*eps)/torch.sqrt(alpha[paso])
        if paso>0:
            var=beta[paso]*(1-acumulado[paso-1])/(1-acumulado[paso]);x=media+var.sqrt()*torch.randn_like(x)
        else:x=media
fig,axs=plt.subplots(1,3,figsize=(11,3))
ruido=torch.randn_like(limpios)
for ax,(datos,titulo) in zip(axs,[(limpios,'Distribución de entrenamiento'),(ruido,'Inicio: ruido'),(x,'Muestras generadas')]):
    ax.scatter(datos[:,0],datos[:,1],s=8,color=AZUL,alpha=.4);ax.set(title=titulo,xlim=(-4,4),ylim=(-4,4),xlabel='Eje 1',ylabel='Eje 2');ax.set_aspect('equal')
mostrar('13-difusion')
print('Aprendimos la distribución, no copiamos el arreglo de entrenamiento. Evalúa visualmente separación y dispersión.')


### Extensiones: modelos preentrenados y tareas más grandes
 El laboratorio principal se ejecuta en CPU y se probó sin descargar pesos externos. Estas extensiones requieren leer las instrucciones del proyecto, instalar sus dependencias y descargar pesos; no se presentan como prácticas ya ejecutadas aquí.

 - **SAM / SAM 2:** segmentación guiada por puntos, cajas o máscaras. Prueba cambiar el punto dentro y fuera del objeto en el [notebook oficial de máscaras](https://colab.research.google.com/github/facebookresearch/segment-anything/blob/main/notebooks/automatic_mask_generator_example.ipynb). Una máscara no dice por sí sola si el objeto está enfermo.
 - **DINOv2:** representación visual auto-supervisada. Usa el [repositorio oficial](https://github.com/facebookresearch/dinov2) para extraer embeddings; compara vecinos de imágenes y explica qué significa similitud.
 - **MAE:** reconstruir parches ocultos para aprender representaciones; consulta el [proyecto original](https://github.com/facebookresearch/mae).
 - **YOLO:** detección con cajas; [tutorial oficial de predicción](https://docs.ultralytics.com/modes/predict/). Compara contar objetos con clasificar una foto completa.
 - **Difusión para imágenes:** [tutorial oficial de Diffusers](https://huggingface.co/docs/diffusers/using-diffusers/write_own_pipeline). Generar una imagen convincente no garantiza exactitud científica.
 - **BERT/GPT y modelos de lenguaje:** el bloque 12 explica atención y posición; entrenar el ejemplo pequeño no equivale a entrenar un LLM. Consulta [Transformers](https://huggingface.co/docs/transformers/index).


## 14. Proyecto: del modelo al impacto
 **El problema:** Una cooperativa considera usar un clasificador como apoyo para revisar granos. ¿Cómo evaluamos valor antes de automatizar?

 **Vamos a hacer:** Comparar siempre revisar manualmente con una política de revisión basada en probabilidades; calcular una matriz de costos ilustrativa.

 **Antes de ejecutar:** Cambia el costo de aceptar un defectuoso y el umbral. ¿Sigue siendo conveniente la misma política?

 **Cómo comprobarlo:** Costo y calidad cambian con la decisión. Los datos y precios son ficticios; no prueban retorno comercial. Los casos reales exigen validación y supervisión.


In [ ]:
# Positivo = defectuoso. Una probabilidad alta dispara revisión manual.
real_defecto=np.array([0,1,0,1,1,0,0,1,0,0])
p_defecto=np.array([.1,.8,.2,.4,.9,.1,.6,.7,.2,.3])
COSTO_REVISION=1.;COSTO_DEFECTO_ACEPTADO=8.
filas=[]
for umbral in [.3,.5,.7,.9]:
    revisar=p_defecto>=umbral
    costo=revisar.sum()*COSTO_REVISION+np.sum((~revisar)&(real_defecto==1))*COSTO_DEFECTO_ACEPTADO
    filas.append([umbral,revisar.sum(),np.sum((~revisar)&(real_defecto==1)),costo])
resultado=pd.DataFrame(filas,columns=['Umbral','Granos revisados','Defectos omitidos','Costo ilustrativo']);display(resultado)
plt.plot(resultado.Umbral,resultado['Costo ilustrativo'],'o-',color=AZUL);plt.axhline(len(real_defecto)*COSTO_REVISION,color=VERDE,ls='--',label='Revisar todos');plt.xlabel('Umbral para revisar');plt.ylabel('Unidades monetarias ficticias');plt.title('La decisión de negocio también tiene un parámetro');plt.legend();mostrar('14-impacto')
print('Supuesto del ejercicio: la revisión manual detecta todos los defectos. En un proyecto real, esa revisión también debe evaluarse.')


## Para cerrar: explica, no solo ejecuta
 Para la conversación de clase, revisa un gráfico, una métrica de prueba y una comparación con una solución simple. Explica qué cambiaría si los datos vinieran de otra finca, otro año u otra cámara. Distingue lo que el modelo calcula de lo que tú decides hacer con el resultado.

 ### Fuentes primarias y rutas de profundización
 - [MIT 6.S191](https://introtodeeplearning.com/): referencia para combinar concepto, laboratorio y proyecto. Este cuaderno es original para UTP, no un material oficial del MIT.
 - [Stanford CS231n](https://cs231n.github.io/): fundamentos visuales de redes y visión.
 - [scikit-learn: evitar fugas de datos](https://scikit-learn.org/stable/common_pitfalls.html).
 - [scikit-learn: agrupación](https://scikit-learn.org/stable/modules/clustering.html).
 - [PyTorch: entrenamiento](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html).
 - [XGBoost](https://xgboost.readthedocs.io/en/stable/python/python_api.html), [LightGBM](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRegressor.html), [SHAP](https://shap.readthedocs.io/en/stable/generated/shap.TreeExplainer.html).
 - [Attention Is All You Need](https://arxiv.org/abs/1706.03762) y [DDPM](https://arxiv.org/abs/2006.11239).

 **Reproducibilidad:** semilla 42. La celda de preparación imprime versiones; las métricas pueden variar entre versiones. Las imágenes de salida son resultados de esta ejecución y se regeneran al ejecutar las celdas.
